In [1]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split, StratifiedKFold,GroupShuffleSplit
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


## set up pyspark session

In [26]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

## set up config

In [27]:
# set up config
model_train_date_str = "2024-09-01"
train_test_period_months = 12
oot_period_months = 2
train_test_ratio = 0.8

config = {}
config["model_train_date_str"] = model_train_date_str
config["train_test_period_months"] = train_test_period_months
config["oot_period_months"] =  oot_period_months
config["model_train_date"] =  datetime.strptime(model_train_date_str, "%Y-%m-%d")
config["oot_end_date"] =  config['model_train_date'] - timedelta(days = 1)
config["oot_start_date"] =  config['model_train_date'] - relativedelta(months = oot_period_months)
config["train_test_end_date"] =  config["oot_start_date"] - timedelta(days = 1)
config["train_test_start_date"] =  config["oot_start_date"] - relativedelta(months = train_test_period_months)
config["train_test_ratio"] = train_test_ratio 


pprint.pprint(config)

{'model_train_date': datetime.datetime(2024, 9, 1, 0, 0),
 'model_train_date_str': '2024-09-01',
 'oot_end_date': datetime.datetime(2024, 8, 31, 0, 0),
 'oot_period_months': 2,
 'oot_start_date': datetime.datetime(2024, 7, 1, 0, 0),
 'train_test_end_date': datetime.datetime(2024, 6, 30, 0, 0),
 'train_test_period_months': 12,
 'train_test_ratio': 0.8,
 'train_test_start_date': datetime.datetime(2023, 7, 1, 0, 0)}


## get label store

In [28]:
# connect to label store
folder_path = "datamart/gold/label_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
label_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",label_store_sdf.count())

label_store_sdf.show()

row_count: 8974
+--------------------+-----------+-----+----------+-------------+
|             loan_id|Customer_ID|label| label_def|snapshot_date|
+--------------------+-----------+-----+----------+-------------+
|CUS_0x1037_2023_0...| CUS_0x1037|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1069_2023_0...| CUS_0x1069|    0|30dpd_6mob|   2023-07-01|
|CUS_0x114a_2023_0...| CUS_0x114a|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1184_2023_0...| CUS_0x1184|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1297_2023_0...| CUS_0x1297|    1|30dpd_6mob|   2023-07-01|
|CUS_0x12fb_2023_0...| CUS_0x12fb|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1325_2023_0...| CUS_0x1325|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1341_2023_0...| CUS_0x1341|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1375_2023_0...| CUS_0x1375|    1|30dpd_6mob|   2023-07-01|
|CUS_0x13a8_2023_0...| CUS_0x13a8|    0|30dpd_6mob|   2023-07-01|
|CUS_0x13ef_2023_0...| CUS_0x13ef|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1440_2023_0...| CUS_0x1440|    0|30dpd_6mob|   2023-0

In [29]:
# extract label store
labels_sdf = label_store_sdf.filter((col("snapshot_date") >= config["train_test_start_date"]) & (col("snapshot_date") <= config["oot_end_date"]))

print("extracted labels_sdf", labels_sdf.count(), config["train_test_start_date"], config["oot_end_date"])

extracted labels_sdf 6961 2023-07-01 00:00:00 2024-08-31 00:00:00


## get features

In [30]:
# connect to feature store
folder_path = "datamart/gold/feature_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
features_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",features_store_sdf.count())

features_store_sdf.show()

row_count: 11974
+-----------+--------------------+---------------+------+--------+-------------------------+-------------+-------------+---------------------+-----------------+---------------+-------------+-----------+--------------------+-------------------+----------------------+--------------------+--------------------+----------+----------------+------------------------+---------------------+-------------------+-----------------------+--------------------+---------------+-------------------------+---+-------------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+---------------------------+-------------------+--------------------+--------------------------+-----------------------+--------------------+--------------+-------------------+-----------------------+-------------------+----------------------------+--------+----------------+----------------+-----------------------+
|Customer_ID|     

In [31]:
# extract features from feature store
features_sdf = features_store_sdf.filter((col("snapshot_date") >= config["train_test_start_date"]) & (col("snapshot_date") <= config["oot_end_date"]))

print("extracted features_sdf", features_sdf.count(), config["train_test_start_date"], config["oot_end_date"])

extracted features_sdf 6937 2023-07-01 00:00:00 2024-08-31 00:00:00


## prepare data for modeling

In [32]:
feat = (features_store_sdf
        .withColumnRenamed("snapshot_date", "feature_snapshot_date")
        .drop("Customer_ID"))

data_pdf = labels_sdf.join(feat, on="loan_id", how="left").toPandas()

In [39]:
data_pdf

,loan_id,Customer_ID,label,label_def,snapshot_date,loan_start_date,tenure,loan_amt,Scheduled_Installment_Amt,feature_snapshot_date,...,Delayed_Payment_Rate,Post_EMI_Slack,Loan_Type_Diversity,Credit_Utilization_Band,Credit_History_Band,Recent_Credit_Expansion_Flag,Age_Band,Occupation_Group,Clickstream_Mean,Clickstream_Vector_Norm
0,CUS_0x1037_2023_01_01,CUS_0x1037,0,30dpd_6mob,2023-07-01,2023-01-01,10,10000.0,0.0,2023-01-01,...,3.7500,1052.62,3,High,Long_Tenure,1,40to59,Professional,55.90,462.96
1,CUS_0x1069_2023_01_01,CUS_0x1069,0,30dpd_6mob,2023-07-01,2023-01-01,10,10000.0,0.0,2023-01-01,...,NaN,4659.56,3,Low,Long_Tenure,1,25to39,Professional,81.25,605.00
2,CUS_0x114a_2023_01_01,CUS_0x114a,0,30dpd_6mob,2023-07-01,2023-01-01,10,10000.0,0.0,2023-01-01,...,1.0000,1210.16,2,Low,Long_Tenure,1,40to59,Professional,108.35,723.95
3,CUS_0x1184_2023_01_01,CUS_0x1184,0,30dpd_6mob,2023-07-01,2023-01-01,10,10000.0,0.0,2023-01-01,...,3.0000,1354.01,3,Low,Long_Tenure,1,40to59,Professional,105.40,576.99
4,CUS_0x1297_2023_01_01,CUS_0x1297,1,30dpd_6mob,2023-07-01,2023-01-01,10,10000.0,0.0,2023-01-01,...,2.6667,4585.23,6,Low,Established,1,40to59,Business,97.80,564.66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6956,CUS_0xdf6_2023_09_01,CUS_0xdf6,0,30dpd_6mob,2024-03-01,2023-09-01,10,10000.0,0.0,2023-09-01,...,NaN,2449.47,0,Medium,Long_Tenure,1,40to59,Trade,107.67,509.71
6957,CUS_0xe23_2023_09_01,CUS_0xe23,0,30dpd_6mob,2024-03-01,2023-09-01,10,10000.0,0.0,2023-09-01,...,14.0000,1414.28,1,Medium,Long_Tenure,1,25to39,Creative,105.76,501.17
6958,CUS_0xe4e_2023_09_01,CUS_0xe4e,0,30dpd_6mob,2024-03-01,2023-09-01,10,10000.0,0.0,2023-09-01,...,NaN,8738.14,0,Medium,Long_Tenure,1,Under25,Professional,110.10,514.69
6959,CUS_0xedd_2023_09_01,CUS_0xedd,0,30dpd_6mob,2024-03-01,2023-09-01,10,10000.0,0.0,2023-09-01,...,3.2500,10107.61,4,Medium,Long_Tenure,0,40to59,Professional,111.34,519.29


In [44]:
# split data into train - test - oot
oot_pdf = data_pdf[(data_pdf['snapshot_date'] >= config["oot_start_date"].date()) & (data_pdf['snapshot_date'] <= config["oot_end_date"].date())]
train_test_pdf = data_pdf[(data_pdf['snapshot_date'] >= config["train_test_start_date"].date()) & (data_pdf['snapshot_date'] <= config["train_test_end_date"].date())]

exclude = ['loan_id', 'snapshot_date', 'label', 'feature_snapshot_date', 'Customer_ID',
           'label_def', 'loan_start_date', 'Occupation', 'Type_of_Loan']
feature_cols = [c for c in data_pdf.columns if c not in exclude]
cat_cols = [c for c in feature_cols if data_pdf[c].dtype == "object"]
num_cols = [c for c in feature_cols if c not in cat_cols]

X_oot = oot_pdf[feature_cols]
y_oot = oot_pdf["label"]
X_train, X_test, y_train, y_test = train_test_split(
    train_test_pdf[feature_cols], train_test_pdf["label"], 
    test_size= 1 - config["train_test_ratio"],
    random_state=88,     # Ensures reproducibility
    shuffle=True,        # Shuffle the data before splitting
    stratify=train_test_pdf["label"]           # Stratify based on the label column
)


print('X_train', X_train.shape[0])
print('X_test', X_test.shape[0])
print('X_oot', X_oot.shape[0])
print('y_train', y_train.shape[0], round(y_train.mean(),2))
print('y_test', y_test.shape[0], round(y_test.mean(),2))
print('y_oot', y_oot.shape[0], round(y_oot.mean(),2))

X_train

X_train 4766
X_test 1192
X_oot 1003
y_train 4766 0.28
y_test 1192 0.28
y_oot 1003 0.29


,tenure,loan_amt,Scheduled_Installment_Amt,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,...,Delayed_Payment_Rate,Post_EMI_Slack,Loan_Type_Diversity,Credit_Utilization_Band,Credit_History_Band,Recent_Credit_Expansion_Flag,Age_Band,Occupation_Group,Clickstream_Mean,Clickstream_Vector_Norm
137,10,10000.0,0.0,48783.44,4187.29,3.0,7.0,18.0,3.0,11.0,...,5.3333,4107.67,3,Low,Long_Tenure,1,Under25,Unknown,96.05,657.80
4090,10,10000.0,0.0,78999.96,6396.33,8.0,5.0,20.0,4.0,42.0,...,6.7500,6258.07,3,Medium,Established,1,40to59,Trade,116.94,537.85
3489,10,10000.0,0.0,15026.04,1059.17,10.0,5.0,28.0,6.0,19.0,...,2.8333,1019.69,5,Medium,Established,1,25to39,Trade,128.30,667.12
431,10,10000.0,0.0,89122.80,7141.90,7.0,3.0,12.0,1.0,31.0,...,14.0000,7084.14,1,Medium,Long_Tenure,1,40to59,Creative,105.85,624.46
4120,10,10000.0,0.0,16753.63,1564.14,6.0,9.0,23.0,9.0,37.0,...,2.7778,1478.58,5,Medium,Established,1,Under25,Professional,94.64,443.78
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2211,10,10000.0,0.0,14489.93,1151.49,3.0,3.0,18.0,1.0,26.0,...,12.0000,1141.86,1,Medium,Long_Tenure,1,25to39,Creative,92.06,474.89
4145,10,10000.0,0.0,156687.16,13095.26,2.0,3.0,7.0,1.0,0.0,...,5.0000,13014.68,1,High,Long_Tenure,0,25to39,Business,113.72,537.79
2304,10,10000.0,0.0,133670.48,11076.21,3.0,3.0,20.0,4.0,13.0,...,2.2500,10812.63,3,Medium,Long_Tenure,1,40to59,Professional,96.69,471.82
6398,10,10000.0,0.0,17585.29,1571.44,7.0,7.0,33.0,5.0,28.0,...,2.0000,1526.66,3,Low,Established,1,Under25,Professional,97.69,472.24


In [45]:
cat_cols

['Credit_Mix',
 'Payment_of_Min_Amount',
 'Payment_Behaviour',
 'Credit_Utilization_Band',
 'Credit_History_Band',
 'Age_Band',
 'Occupation_Group']

## preprocess data

In [49]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

# fit on train only; test and OOT are transformed with train statistics
transformer_stdscaler = preprocessor.fit(X_train)

X_train_processed = transformer_stdscaler.transform(X_train)
X_test_processed  = transformer_stdscaler.transform(X_test)
X_oot_processed   = transformer_stdscaler.transform(X_oot)

print('X_train_processed', X_train_processed.shape)
print('X_test_processed', X_test_processed.shape)
print('X_oot_processed', X_oot_processed.shape)

pd.DataFrame(X_train_processed, columns=transformer_stdscaler.get_feature_names_out())

X_train_processed (4766, 81)
X_test_processed (1192, 81)
X_oot_processed (1003, 81)


,num__tenure,num__loan_amt,num__Scheduled_Installment_Amt,num__Annual_Income,num__Monthly_Inhand_Salary,num__Num_Bank_Accounts,num__Num_Credit_Card,num__Interest_Rate,num__Num_of_Loan,num__Delay_from_due_date,...,cat__Age_Band_25to39,cat__Age_Band_40to59,cat__Age_Band_60plus,cat__Age_Band_Under25,cat__Occupation_Group_Business,cat__Occupation_Group_Creative,cat__Occupation_Group_Education,cat__Occupation_Group_Professional,cat__Occupation_Group_Trade,cat__Occupation_Group_Unknown
0,0.0,0.0,0.0,-0.039359,0.007524,-0.923253,0.736479,0.401850,-0.206565,-0.680640,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.0,0.0,0.0,0.758436,0.711366,1.020220,-0.239387,0.629550,0.200584,1.406006,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.0,0.0,0.0,-0.930644,-0.989154,1.797609,-0.239387,1.540349,1.014881,-0.142150,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.0,0.0,0.0,1.025706,0.948919,0.631525,-1.215254,-0.281249,-1.020862,0.665583,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,-0.885031,-0.828261,0.242831,1.712346,0.971100,2.236328,1.069450,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4761,0.0,0.0,0.0,-0.944799,-0.959739,-0.923253,-1.215254,0.401850,-1.020862,0.329028,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4762,0.0,0.0,0.0,2.809584,2.845772,-1.311947,-1.215254,-0.850498,-1.020862,-1.421062,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4763,0.0,0.0,0.0,2.201882,2.202465,-0.923253,-1.215254,0.629550,0.200584,-0.546017,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4764,0.0,0.0,0.0,-0.863073,-0.825935,0.631525,0.736479,2.109598,0.607733,0.463650,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


## train model

In [50]:
# Define the XGBoost classifier
xgb_clf = xgb.XGBClassifier(eval_metric='logloss', random_state=88)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=88)

# Define the hyperparameter space to search
param_dist = {
    'n_estimators': [25, 50],
    'max_depth': [2, 3],  # lower max_depth to simplify the model
    'learning_rate': [0.01, 0.1],
    'subsample': [0.6, 0.8],
    'colsample_bytree': [0.6, 0.8],
    'gamma': [0, 0.1],
    'min_child_weight': [1, 3, 5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

# Create a scorer based on AUC score
auc_scorer = make_scorer(roc_auc_score)

# Set up the random search with cross-validation
random_search = RandomizedSearchCV(
    estimator=xgb_clf,
    param_distributions=param_dist,
    scoring=auc_scorer,
    n_iter=10,  # Number of iterations for random search
    cv=3,       # Number of folds in cross-validation
    verbose=1,
    random_state=42,
    n_jobs=-1   # Use all available cores
)

# Perform the random search
random_search.fit(X_train_processed, y_train)

# Output the best parameters and best score
print("Best parameters found: ", random_search.best_params_)
print("Best AUC score: ", random_search.best_score_)

# Evaluate the model on the train set
best_model = random_search.best_estimator_
y_pred_proba = best_model.predict_proba(X_train_processed)[:, 1]
train_auc_score = roc_auc_score(y_train, y_pred_proba)
print("Train AUC score: ", train_auc_score)

# Evaluate the model on the test set
best_model = random_search.best_estimator_
y_pred_proba = best_model.predict_proba(X_test_processed)[:, 1]
test_auc_score = roc_auc_score(y_test, y_pred_proba)
print("Test AUC score: ", test_auc_score)

# Evaluate the model on the oot set
best_model = random_search.best_estimator_
y_pred_proba = best_model.predict_proba(X_oot_processed)[:, 1]
oot_auc_score = roc_auc_score(y_oot, y_pred_proba)
print("OOT AUC score: ", oot_auc_score)

print("TRAIN GINI score: ", round(2*train_auc_score-1,3))
print("Test GINI score: ", round(2*test_auc_score-1,3))
print("OOT GINI score: ", round(2*oot_auc_score-1,3))

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters found:  {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 50, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 0.8}
Best AUC score:  0.7379415027284354
Train AUC score:  0.9020809700245122
Test AUC score:  0.8795009283842644
OOT AUC score:  0.9132398934321788
TRAIN GINI score:  0.804
Test GINI score:  0.759
OOT GINI score:  0.826


## prepare model artefact to save

In [51]:
model_artefact = {}

model_artefact['model'] = best_model
model_artefact['model_version'] = "credit_model_"+config["model_train_date_str"].replace('-','_')
model_artefact['preprocessing_transformers'] = {}
model_artefact['preprocessing_transformers']['stdscaler'] = transformer_stdscaler
model_artefact['data_dates'] = config
model_artefact['data_stats'] = {}
model_artefact['data_stats']['X_train'] = X_train.shape[0]
model_artefact['data_stats']['X_test'] = X_test.shape[0]
model_artefact['data_stats']['X_oot'] = X_oot.shape[0]
model_artefact['data_stats']['y_train'] = round(y_train.mean(),2)
model_artefact['data_stats']['y_test'] = round(y_test.mean(),2)
model_artefact['data_stats']['y_oot'] = round(y_oot.mean(),2)
model_artefact['results'] = {}
model_artefact['results']['auc_train'] = train_auc_score
model_artefact['results']['auc_test'] = test_auc_score
model_artefact['results']['auc_oot'] = oot_auc_score
model_artefact['results']['gini_train'] = round(2*train_auc_score-1,3)
model_artefact['results']['gini_test'] = round(2*test_auc_score-1,3)
model_artefact['results']['gini_oot'] = round(2*oot_auc_score-1,3)
model_artefact['hp_params'] = random_search.best_params_


pprint.pprint(model_artefact)

{'data_dates': {'model_train_date': datetime.datetime(2024, 9, 1, 0, 0),
                'model_train_date_str': '2024-09-01',
                'oot_end_date': datetime.datetime(2024, 8, 31, 0, 0),
                'oot_period_months': 2,
                'oot_start_date': datetime.datetime(2024, 7, 1, 0, 0),
                'train_test_end_date': datetime.datetime(2024, 6, 30, 0, 0),
                'train_test_period_months': 12,
                'train_test_ratio': 0.8,
                'train_test_start_date': datetime.datetime(2023, 7, 1, 0, 0)},
 'data_stats': {'X_oot': 1003,
                'X_test': 1192,
                'X_train': 4766,
                'y_oot': np.float64(0.29),
                'y_test': np.float64(0.28),
                'y_train': np.float64(0.28)},
 'hp_params': {'colsample_bytree': 0.8,
               'gamma': 0.1,
               'learning_rate': 0.1,
               'max_depth': 3,
               'min_child_weight': 1,
               'n_estimators': 50,
        

## save artefact to model bank

In [52]:
# create model_bank dir
model_bank_directory = "model_bank/"

if not os.path.exists(model_bank_directory):
    os.makedirs(model_bank_directory)

In [53]:
# Full path to the file
file_path = os.path.join(model_bank_directory, model_artefact['model_version'] + '.pkl')

# Write the model to a pickle file
with open(file_path, 'wb') as file:
    pickle.dump(model_artefact, file)

print(f"Model saved to {file_path}")


Model saved to model_bank/credit_model_2024_09_01.pkl


## test load pickle and make model inference

In [54]:
# Load the model from the pickle file
with open(file_path, 'rb') as file:
    loaded_model_artefact = pickle.load(file)

y_pred_proba = loaded_model_artefact['model'].predict_proba(X_oot_processed)[:, 1]
oot_auc_score = roc_auc_score(y_oot, y_pred_proba)
print("OOT AUC score: ", oot_auc_score)

print("Model loaded successfully!")

OOT AUC score:  0.9132398934321788
Model loaded successfully!
